In [1]:
import pandas as pd
import numpy as np

# Keep the random numbers the exact same everytime we run it
np.random.seed(0)
n_customers = 1000

# Create a portfolio of 1000 banking customers using a Python Dictionary
mock_banking_data= {
    'Customer_ID' : np.arange(10001, 10001 + n_customers),
    'Tenure_Years' : np.random.randint(1, 15, size = n_customers),
    'Active_Products' : np.random.randint(1, 5, size = n_customers),
    'Current_Balance' : np.random.exponential(scale = 10000, size = n_customers).round(2),
    'Avg_3Month_Balance' : np.random.exponential(scale = 12000, size = n_customers).round(2),
    'Monthly_Transactions_Current' : np.random.randint(2, 50, size = n_customers),
    'Monthly_Transactions_Last_Month' : np.random.randint(5, 50, size = n_customers),
    }

# Turn the dictionary of data into a dataframe
df = pd.DataFrame(mock_banking_data)

# Balance Velocity Calculation (Current Month Balance / 3 Month Average Balance)
# A value below 1.0 means the balance is declining relative to the average
df['Balance_Velocity'] = (df['Current_Balance'] / (df['Avg_3Month_Balance'] + 1)).round(4)

# Calculate the Transaction Drop off (Percentage change in transactions month-over-month)
# A negative value means the customer is transacting less than last month
df['Transaction_Dropoff'] = ((df['Monthly_Transactions_Current'] - df['Monthly_Transactions_Last_Month']) / df['Monthly_Transactions_Last_Month']).round(4)

# Churn Labels

# Declining balance = 40% weight
# Transaction dropoff = 30% weight
# Short tenure = 20% weight
# Only one product = 10% weight

churn_score = (
    (df['Balance_Velocity'] < 0.5).astype(int) * 0.4 +
    (df['Transaction_Dropoff'] < -0.2).astype(int) * 0.3 +
    (df['Tenure_Years'] < 3).astype(int) * 0.2 +
    (df['Active_Products'] == 1).astype(int) * 0.1
)

# A customer with all four risk signals gets a score of 1.0 (maximum risk). A customer with none gets 0.0.
if churn_score.max() > 0:
    churn_prob = churn_score / churn_score.max()
else:
    churn_prob = churn_score

#  (churn_prob ** 2) to ensure only the highest-risk profiles get pushed to Churn
df['Churn_Label'] = (np.random.random(n_customers) < (churn_prob **2)).astype(int)

# View Output

print("--- Our First 5 Banking Customer Profiles ---")
print(df[['Customer_ID', 'Balance_Velocity', 'Transaction_Dropoff', 'Churn_Label']].head())

print(f"\nTotal customers : {len(df)}")
print(f"Churned         : {df['Churn_Label'].sum()} ({df['Churn_Label'].mean()*100:.1f}%)")
print(f"Retained        : {(df['Churn_Label']==0).sum()}")

--- Our First 5 Banking Customer Profiles ---
   Customer_ID  Balance_Velocity  Transaction_Dropoff  Churn_Label
0        10001            5.5686               3.4545            0
1        10002            2.0329              -0.8444            0
2        10003            0.9597               1.3500            0
3        10004            0.0053               5.1429            0
4        10005            0.0086               0.6667            0

Total customers : 1000
Churned         : 165 (16.5%)
Retained        : 835


In [2]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE


features = ['Tenure_Years', 'Active_Products', 'Balance_Velocity', 'Transaction_Dropoff' ]
X = df[features]
y = df['Churn_Label']

# Split the dataset into training and testing sets
# stratify = y ensures the train/test split maintains
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42, stratify = y)


# Balance the skewed classes mathematically using SMOTE
smote = SMOTE(random_state = 42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


# Initialize and train XGBoost tree engine
model = XGBClassifier(n_estimators = 100, max_depth = 4, learning_rate = 0.1, random_state = 42)
model.fit(X_train_resampled, y_train_resampled)


# Calculate and append continuous probability percentages
df['Churn_Probability'] = model.predict_proba(X)[:, 1].round(4)


# View calculated risk scores
print("--- Machine Learning Predictions Appended to Portfolio ---")
print(df[['Customer_ID', 'Balance_Velocity', 'Churn_Probability', 'Churn_Label']].head())


--- Machine Learning Predictions Appended to Portfolio ---
   Customer_ID  Balance_Velocity  Churn_Probability  Churn_Label
0        10001            5.5686             0.0255            0
1        10002            2.0329             0.1447            0
2        10003            0.9597             0.0458            0
3        10004            0.0053             0.1544            0
4        10005            0.0086             0.5027            0


In [3]:
# Check accuracy of the model
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
print("\n--- Model Performance ---")
print(classification_report(y_test, y_pred,
      target_names=['Retained', 'Churned']))


--- Model Performance ---
              precision    recall  f1-score   support

    Retained       0.92      0.78      0.84       167
     Churned       0.37      0.67      0.48        33

    accuracy                           0.76       200
   macro avg       0.65      0.72      0.66       200
weighted avg       0.83      0.76      0.78       200



In [4]:
# Filter for high-risk accounts
high_risk_pipeline = df[df['Churn_Probability'] > 0.1].copy()

# Build the structural engineering prompt template

def generate_retention_prompt(row):
    prompt = f"""
    [ROLE]: Senior Financial Retention Strategist at a Big 5 Bank
    [CLIENT METRICS]:
    - Customer Account ID: {int(row['Customer_ID'])}
    - Account Tenure: {int(row['Tenure_Years'])} Years
    - Estimated Attrition Risk: {row['Churn_Probability'] * 100:.1f}%
    - Asset Balance Velocity: {row['Balance_Velocity']:.2f}
    - Monthly Transaction Volume Drop-off: {row['Transaction_Dropoff'] * 100:.1f}%

    [TASK]: Synthesize a highly personalized retention outreach blueprint.
    If Balance Velocity is low, offer an exclusive high-interest TFSA/GIC rate match.
    If Transaction Drop-off is high, offer a 12-month annual fee waiver on their premium credit card product.
    Keep the tone professional, empathetic, and concise
    """
    return prompt.strip()

# Map the function to create a new column
high_risk_pipeline['AI_Prompt'] = high_risk_pipeline.apply(generate_retention_prompt, axis = 1)

# Save the processed pipeline to a clean CSV file
high_risk_pipeline.to_csv('high_risk_attrition_portfolio.csv', index = False)

print(f"--- GenAI Data Pipeline Staging Complete ---")
print(f"Successfully isolated {len(high_risk_pipeline)} high-risk profiles and exported to CSV.")


# Save the full master file for Tableau visualizations
df.to_csv('banking_churn_tableau_master.csv', index = False)
print("Master Tableau data file successfully exported!")

--- GenAI Data Pipeline Staging Complete ---
Successfully isolated 612 high-risk profiles and exported to CSV.
Master Tableau data file successfully exported!


In [6]:
# Install Google Geminai quietly (uncomment if running for the first time)
# %pip install -q google-genai python-dotenv

In [5]:
import os
from google import genai
from dotenv import load_dotenv, find_dotenv

# Hidden API Key
load_dotenv(find_dotenv())

# Start the Gemini AI client
client = genai.Client()

# Test with the top 3 high-risk clients
# (Remove .head(3) below to execute full portfolio pipeline)
test_pipeline = high_risk_pipeline.head(3).copy()

print("--- Securely Connecting to Gemini API ---")
ai_emails = []

# Loop through our 3 target customers row-by-row

for index, row in test_pipeline.iterrows():
    customer_id = int(row['Customer_ID'])
    print(f"Processing customer {customer_id}...")

    prompt_payload = row['AI_Prompt']

    # Call the live model
    response = client.models.generate_content(model = 'gemini-2.5-flash',
    contents = prompt_payload,)

    ai_emails.append(response.text)

test_pipeline['AI_Retention_Email'] = ai_emails
test_pipeline.to_csv('ai_retention_outputs.csv', index = False)

print("\n--- Success! Check 'ai_retention_outputs.csv' ---")



--- Securely Connecting to Gemini API ---
Processing customer 10002...
Processing customer 10004...
Processing customer 10005...

--- Success! Check 'ai_retention_outputs.csv' ---
